In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from keras.layers import InputLayer,Input
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.metrics import R2Score

SEED = 29

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [2]:
df = pd.read_parquet("train.parquet")

In [24]:
seqs = np.zeros((517,1000,32))
for i in range(517):
   seqs[i] = df[df["seq_ix"] == i].drop(columns= ["need_prediction","seq_ix","step_in_seq"]).values

diff_seqs = np.zeros((517,999,32))
for i in range(517):
   for j in range(999):
    diff_seqs[i][j] = seqs[i][j+1] - seqs[i][j]



In [30]:
input_to_encoder_seqs = np.zeros((517,99,32))
for i in range(517):
   input_to_encoder_seqs[i] = diff_seqs[i][0:99][:]


In [32]:
input_to_encoder_seqs

array([[[-1.16319157, -0.92613841, -2.44821508, ..., -0.67959365,
         -0.15328736, -0.6754949 ],
        [ 1.11859414,  0.82715497,  1.94996668, ...,  0.93355488,
          0.09312193,  0.75437739],
        [-1.13977574, -0.73109567, -1.18114455, ..., -1.07063839,
         -0.08327062, -0.94743139],
        ...,
        [ 0.20882979, -0.06350725,  0.67784145, ..., -0.07880257,
          0.07561443,  0.45249372],
        [ 0.0198048 , -0.0964024 , -0.47905714, ..., -0.14770322,
         -0.27528824,  0.0323199 ],
        [ 0.21118096, -0.31189698,  0.25506207, ..., -0.22050519,
         -0.39178196,  0.50283418]],

       [[ 0.75564486, -1.25884416,  0.11113814, ...,  0.60734714,
         -1.09694598,  0.68538561],
        [ 2.28668812, -0.02808436,  1.34781734, ...,  1.1415971 ,
         -1.10858253,  0.61677959],
        [-1.65316873,  1.44451637, -1.12696384, ..., -4.61475893,
          0.58186639,  1.85541602],
        ...,
        [-1.35998946,  0.84315429, -0.80380754, ...,  

In [31]:
output_of_decoder_seqs = np.zeros((517,900,32))
for i in range(517):
   output_of_decoder_seqs[i] =  diff_seqs[i][99:999][:]
   


In [33]:
output_of_decoder_seqs

array([[[-3.95611399e-01,  4.51547791e-01, -3.55731204e-01, ...,
          2.92687070e-01,  5.75685399e-01, -8.28283053e-01],
        [ 3.97051029e-01, -4.85717354e-01,  3.63542861e-01, ...,
         -4.74549251e-02, -5.81176351e-01,  8.05795071e-01],
        [-1.20855108e+00, -9.34230717e-01, -1.74459458e+00, ...,
         -8.70757990e-01,  1.12410920e-01, -1.25291946e+00],
        ...,
        [-1.04319146e+00, -1.19102127e+00, -1.47983011e+00, ...,
         -4.13547292e-01,  2.84914737e-02, -1.59623762e+00],
        [-3.76286253e-02,  2.34055906e+00,  7.32877559e-01, ...,
          1.46050153e+00,  1.81531260e+00, -3.33829756e-01],
        [ 2.05896350e-01, -3.12753512e-01,  2.71576855e-01, ...,
          4.25267066e-02, -3.54834820e-01,  4.78045578e-01]],

       [[ 4.05544854e-01, -3.43379063e-01,  5.10304587e-01, ...,
         -1.96783695e+00, -6.61778002e-01,  1.96476394e+00],
        [-1.83502689e+00, -1.27170012e+00, -1.46582662e+00, ...,
          7.54813417e-01,  5.79477528e

In [27]:
import tensorflow.keras.backend as K
def r2_keras(y_true, y_pred):
    """
    Custom R2-score metric for 3D tensors (batch, time_steps, features).
    Flattens the time steps dimension before calculating R2.
    """
    # Reshape tensors from (batch, time_steps, features) to (batch * time_steps, features)
    y_true_reshaped = K.reshape(y_true, (-1, K.shape(y_true)[-1]))
    y_pred_reshaped = K.reshape(y_pred, (-1, K.shape(y_pred)[-1]))

    # Calculate Sum of Squares Residual (SS_res)
    SS_res = K.sum(K.square(y_true_reshaped - y_pred_reshaped))

    # Calculate Total Sum of Squares (SS_tot)
    # Mean is calculated across all flattened samples
    SS_tot = K.sum(K.square(y_true_reshaped - K.mean(y_true_reshaped)))

    # Calculate R2 score, add epsilon to prevent division by zero
    r2 = 1 - SS_res / (SS_tot + K.epsilon())

    return r2

In [34]:
class AutoEncoder(Model):
  def __init__(self,input_shape):
    self.num_neurons = [40,40,40]
    
    self.enc_dims = 64
    self.decoder_neurons = self.num_neurons[-1]
    self.to_predict = 900
    # self.encoder_layers = []
    # self.decoder_layers = []
    super(AutoEncoder,self).__init__()
    


    self.encoder_lstm_1 = layers.LSTM(self.num_neurons[0],
                                          return_sequences=True,
                                          name="encoder_lstm_1",
                                          recurrent_dropout = 0.3) 
    self.encoder_lstm_middle = layers.LSTM(self.num_neurons[1],
                                          return_sequences=True,
                                          name="encoder_lstm_middle")
    self.projector_layer = layers.Dense(self.num_neurons[1],activation="linear",name = "projector")
    
    self.encoder_lstm_2 = layers.LSTM(self.num_neurons[2],
                                          return_sequences=True,
                                          return_state=True,
                                          name="encoder_lstm_2")

    self.decoder_lstm = layers.LSTM(self.decoder_neurons,
                                         return_sequences=True,
                                         return_state=True,
                                         name="decoder_lstm")
    
    self.norm_1 = layers.LayerNormalization(name = "norm1")
    self.norm_2 = layers.LayerNormalization(name = "norm2")
    self.norm_3 = layers.LayerNormalization(name = "norm3")
    self.norm_4 = layers.LayerNormalization(name = "norm4")

    self.DenseQ = layers.Dense(self.enc_dims,name = "Query_Dense")
    self.DenseK = layers.Dense(self.enc_dims,name = "Query_Key")
    self.DenseV = layers.Dense(self.enc_dims,name = "Query_Value")
    self.dropout = layers.Dropout(0.3)

    # self.FFn_neurons = [24,36,self.enc_dims]

    # self.FFn_Dense1 = layers.Dense()

    num_heads = 4
    self.self_attention_layer = layers.MultiHeadAttention(name = "self_attention",num_heads = num_heads,key_dim = self.enc_dims//num_heads)
    self.cross_attention_layer = layers.MultiHeadAttention(name="attention",num_heads = num_heads,key_dim= self.enc_dims//num_heads)
    
    self.concat_layer = layers.Concatenate(axis=-1, name="concat_attention")
    self.output_dense_layer = layers.Dense(input_shape[1], name="output_dense",activation="linear") 
    self.output_time_dist = layers.TimeDistributed(self.output_dense_layer, name="output_time_distributed") 



    inputs = tf.keras.Input(input_shape,name = "init_input")  
    x1 = self.encoder_lstm_1(inputs)
    x = self.encoder_lstm_middle(x1)
    x1_proj = self.projector_layer(x1)
    x = layers.Add(name = "skip_conn")([x1_proj,x])
    x = self.norm_1(x)
    
    lstm_out , h , c = self.encoder_lstm_2(x)
    states = [h,c]

    key = self.DenseK(lstm_out)
    query = self.DenseQ(lstm_out)
    val = self.DenseV(lstm_out)

    attention_context_for_enc = self.self_attention_layer(query =query , value = val,key =key)
    encoder_out = self.concat_layer([attention_context_for_enc,lstm_out])
    encoder_out = self.norm_2(encoder_out)


    self.encoder_model = Model(inputs = inputs,outputs = [encoder_out] + states , name = "ENCODER")

    target = tf.keras.Input(shape = (self.to_predict,input_shape[1]))
    decoder_lstm_outputs, _, _ = self.decoder_lstm(target,initial_state=states)



    attention_context = self.cross_attention_layer(decoder_lstm_outputs, encoder_out,encoder_out)
    decoder_combined_context = self.concat_layer([attention_context, decoder_lstm_outputs])
    decoder_combined_context = self.norm_3(decoder_combined_context)
    decoder_combined_context = self.dropout(decoder_combined_context)
    decoder_pred_outputs = self.output_time_dist(decoder_combined_context)


    self.training_model = Model(inputs=[inputs, target],outputs=decoder_pred_outputs)

    #----------------------------------------------------------------------------------------------------
    inf_decoder_input_prev = Input(shape=(1, input_shape[1]), name="inf_decoder_input") # Previous prediction
    inf_decoder_state_h = Input(shape=(self.decoder_neurons,), name="inf_state_h")          # Previous h state
    inf_decoder_state_c = Input(shape=(self.decoder_neurons,), name="inf_state_c")          # Previous c state
    inf_decoder_states_prev = [inf_decoder_state_h, inf_decoder_state_c]                 # Combined previous states
    inf_encoder_outputs = Input(shape=(input_shape[0], self.num_neurons[-1]+self.enc_dims), name="inf_encoder_outputs") # ALL encoder outputs for attention

        # Connect layers for one step using shared layer instances:
    inf_dec_lstm_out, inf_state_h_out, inf_state_c_out = self.decoder_lstm(
        inf_decoder_input_prev, initial_state=inf_decoder_states_prev # Initialize with previous states
    )
    inf_attention_context = self.cross_attention_layer(inf_dec_lstm_out, inf_encoder_outputs,inf_encoder_outputs) # Calculate attention for this step
    inf_combined_context = self.concat_layer([inf_attention_context, inf_dec_lstm_out])    # Combine
    inf_decoder_pred = self.output_time_dist(inf_combined_context)                       # Predict (shape=(batch, 1, features))

        # Define the single-step inference decoder model
    self.inf_decoder_model = Model(
        inputs=[inf_decoder_input_prev, inf_decoder_states_prev, inf_encoder_outputs],
        outputs=[inf_decoder_pred] + [inf_state_h_out, inf_state_c_out], # Output prediction AND next states
        name="inf_decoder"
    )      

    
                            
  

  def train(self,X_train,y_train):
    last_known_step = X_train[:, -1:, :] 
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_r2_keras',  
        factor=0.5,          
        patience=2,          
        verbose=1,           
        mode='max',         
        min_delta=0.002,      
        cooldown=2,          
        min_lr=0.000002        
    )
    future_inputs = y_train[:, :-1, :]

    # learning_scheduler = tf.keras.optimizers.schedules.ExponentialDecay(
    #     0.01,
    #     decay_steps=10000,
    #     decay_rate=0.96,
    #     staircase=True)
    
    # 3. CREATE THE NEW DECODER INPUT VARIABLE
    # This is the "shifted" sequence for teacher forcing
    # Shape: (batch_size, 899, 32)
    decoder_input_data = np.concatenate([last_known_step, future_inputs], axis=1)
    
    # 4. THE TARGET IS JUST THE ORIGINAL Y_TRAIN
    decoder_target_data = y_train
    optim = tf.keras.optimizers.AdamW(learning_rate=0.01,clipnorm = 1.0)
    self.training_model.compile(optimizer = optim,loss= "mse",metrics=[r2_keras])
    self.training_model.summary()
    self.training_model.fit([X_train,decoder_input_data],
                            y_train,batch_size = 32, 
                            epochs = 25,
                            verbose = True, 
                            validation_split = 0.2,
                            callbacks = [reduce_lr]
                            )
    self.training_model.save_weights("C:/CodingWorkspace/Python/WunderRNN/AE_wts_diffs.weights.h5")
    


In [35]:
autoenc = AutoEncoder([99,32])
autoenc.train(X_train = input_to_encoder_seqs,y_train=output_of_decoder_seqs)

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ init_input          │ (None, 99, 32)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_1      │ (None, 99, 40)    │     11,680 │ init_input[0][0]  │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ projector (Dense)   │ (None, 99, 40)    │      1,640 │ encoder_lstm_1[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_middle │ (None, 99, 40)    │     12,960 │ encoder_lstm_1[0… │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ skip_conn (Add)     │ (None, 99, 40)    │          0 │ projector[0][0],  │
│                     │                   │            │ encoder_lstm_mid… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ norm1               │ (None, 99, 40)    │         80 │ skip_conn[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_2      │ [(None, 99, 40),  │     12,960 │ norm1[0][0]       │
│ (LSTM)              │ (None, 40),       │            │                   │
│                     │ (None, 40)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Query_Key (Dense)   │ (None, 99, 64)    │      2,624 │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Query_Dense (Dense) │ (None, 99, 64)    │      2,624 │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Query_Value (Dense) │ (None, 99, 64)    │      2,624 │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ self_attention      │ (None, 99, 64)    │     16,640 │ Query_Key[0][0],  │
│ (MultiHeadAttentio… │                   │            │ Query_Dense[0][0… │
│                     │                   │            │ Query_Value[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_attention    │ (None, 1, 80)     │          0 │ self_attention[0… │
│ (Concatenate)       │                   │            │ encoder_lstm_2[0… │
│                     │                   │            │ attention[0][0],  │
│                     │                   │            │ decoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_7       │ (None, 900, 32)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 1, 40),   │     11,680 │ input_layer_7[0]… │
│                     │ (None, 40),       │            │ encoder_lstm_2[0… │
│                     │ (None, 40)]       │            │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ norm2               │ (None, 99, 104)   │        208 │ concat_attention… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 1, 40)     │     18,664 │ decoder_lstm[0][… │
│ (MultiHeadAttentio… │                   │            │ norm2[0][0],      │
│                     │                   │            │ norm2[0][0]     

 Total params: 97,136 (379.44 KB)

 Trainable params: 97,136 (379.44 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 54s 2s/step - loss: 1.9707 - r2_keras: -0.4683 - val_loss: 1.2979 - val_r2_keras: 0.0985 - learning_rate: 0.0100
Epoch 2/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 10s 722ms/step - loss: 1.1625 - r2_keras: 0.1420 - val_loss: 1.0135 - val_r2_keras: 0.3023 - learning_rate: 0.0100
Epoch 3/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 11s 847ms/step - loss: 0.9325 - r2_keras: 0.3116 - val_loss: 0.9114 - val_r2_keras: 0.3756 - learning_rate: 0.0100
Epoch 4/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 11s 810ms/step - loss: 0.8519 - r2_keras: 0.3708 - val_loss: 0.8336 - val_r2_keras: 0.4297 - learning_rate: 0.0100
Epoch 5/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 10s 751ms/step - loss: 0.7991 - r2_keras: 0.4094 - val_loss: 0.7886 - val_r2_keras: 0.4603 - learning_rate: 0.0100
Epoch 6/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 700ms/step - loss: 0.7682 - r2_keras: 0.4319 - val_loss: 0.7804 - val_r2_keras: 0.4655 - learning_rate: 0.0100
Epoch 7/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 694ms/step - loss: 0.7494 - r2_keras: 0.4458